# Project 13 — Building an Autograd Playground | PyTorch Fundamentals II — Autograd | Automatic Differentiation: Backpropagation in Practice

## Project Description

**Objective**: Develop an intuitive understanding of PyTorch's automatic differentiation engine.

Create a Jupyter notebook or Python script that:

1. Demonstrates computational graph construction with simple mathematical expressions.
1. Shows how `requires_grad` affects graph creation.
1. Computes gradients for several scalar functions and verifies them analytically.
1. Demonstrates gradient accumulation and the importance of clearing gradients.
1. Explores `grad_fn` for different operations.
1. Compares computations performed:
   - with `Autograd` enabled,
   - inside `torch.no_grad()`,
   - after using `detach()`.
1. Includes brief explanations of each experiment and what it reveals about Autograd.

The emphasis is on understanding, not on training a neural network.

## My Solution

### Setup & Environment

In [1]:
import torch

print(f"PyTorch Version: {torch.__version__}")

PyTorch Version: 2.13.0+cpu


### Computational Graph Construction & `requires_grad`

PyTorch constructs a dynamic Directed Acyclic Graph (DAG) on the fly during the forward pass. Tensors created by the user are leaf nodes, while tensors resulting from operations are intermediate nodes.

In [4]:
# Experiment 1: Tracking vs Non-Tracking
x = torch.tensor(2.0)                 # Default: requires_grad=False
y = torch.tensor(3.0, requires_grad=True) # Explicit tracking

z1 = x * 2 + 1
z2 = y * 2 + 1

print("--- Experiment 1: requires_grad Effect ---")
print(f"x (requires_grad={x.requires_grad}): is_leaf={x.is_leaf}, grad_fn={x.grad_fn}")
print(f"z1 (from x): requires_grad={z1.requires_grad}, grad_fn={z1.grad_fn}")
print(f"y (requires_grad={y.requires_grad}): is_leaf={y.is_leaf}, grad_fn={y.grad_fn}")
print(f"z2 (from y): requires_grad={z2.requires_grad}, grad_fn={z2.grad_fn}")

--- Experiment 1: requires_grad Effect ---
x (requires_grad=False): is_leaf=True, grad_fn=None
z1 (from x): requires_grad=False, grad_fn=None
y (requires_grad=True): is_leaf=True, grad_fn=None
z2 (from y): requires_grad=True, grad_fn=<AddBackward0 object at 0x000002026A2331F0>


### Exploring `grad_fn` Across Operations

Every output tensor resulting from an operation on tracked inputs holds a reference to a `grad_fn` object. This object knows how to compute the local derivative during the backward pass.

In [11]:
a = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)

# Math expressions
c = a + b
d = a * b
e = d ** 2
f = torch.sin(e)

print("\n--- Experiment 2: Inspecting grad_fn ---")
print(f"Addition (c):       {c.grad_fn}")
print(f"Multiplication (d): {d.grad_fn}")
print(f"Power (e):          {e.grad_fn}")
print(f"Sin (f):            {f.grad_fn}")

# Walking back the computational graph manually
print(f"\nTraceback from 'f': {f.grad_fn} -> next_functions: {f.grad_fn.next_functions} {f.grad_fn.next_functions[0][0].next_functions}")


--- Experiment 2: Inspecting grad_fn ---
Addition (c):       <AddBackward0 object at 0x000002026A89A0E0>
Multiplication (d): <MulBackward0 object at 0x000002025C7DA0E0>
Power (e):          <PowBackward0 object at 0x000002026A8214B0>
Sin (f):            <SinBackward0 object at 0x000002025C94B3A0>

Traceback from 'f': <SinBackward0 object at 0x000002025C94B3A0> -> next_functions: ((<PowBackward0 object at 0x000002026A8214B0>, 0),) ((<MulBackward0 object at 0x000002025C7DA0E0>, 0),)


### Gradient Computation & Analytical Verification

Autograd computes partial derivatives using the chain rule. Below, we compute gradients for two scalar functions and compare them against hand-derived derivatives.

#### Function A: Single-Variable Polynomial

$$f(x) = 3x^2 + 2x + 1 \implies f'(x) = 6x + 2$$

In [13]:
x = torch.tensor(4.0, requires_grad=True)
y = 3 * (x ** 2) + 2 * x + 1

y.backward()

# Analytical derivative at x = 4
analytical_grad_x = 6 * 4.0 + 2

print("\n--- Experiment 3A: Single-Variable Derivative ---")
print(f"Autograd df/dx:   {x.grad.item()}")
print(f"Analytical df/dx: {analytical_grad_x}")
assert torch.isclose(x.grad, torch.tensor(analytical_grad_x))


--- Experiment 3A: Single-Variable Derivative ---
Autograd df/dx:   26.0
Analytical df/dx: 26.0


#### Function B: Multi-Variable Function

$$
f(a, b) = a^3 b - b^2 \implies \frac{\partial f}{\partial a} = 3a^2b, \quad \frac{\partial f}{\partial b} = a^3 - 2b
$$

In [14]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(5.0, requires_grad=True)

f = (a ** 3) * b - (b ** 2)
f.backward()

# Analytical gradients at a=2, b=5
analytical_grad_a = 3 * (2.0 ** 2) * 5.0      # 60.0
analytical_grad_b = (2.0 ** 3) - 2 * 5.0      # -2.0

print("\n--- Experiment 3B: Multi-Variable Partial Derivatives ---")
print(f"Autograd df/da:   {a.grad.item()} | Analytical: {analytical_grad_a}")
print(f"Autograd df/db:   {b.grad.item()} | Analytical: {analytical_grad_b}")


--- Experiment 3B: Multi-Variable Partial Derivatives ---
Autograd df/da:   60.0 | Analytical: 60.0
Autograd df/db:   -2.0 | Analytical: -2.0


### Gradient Accumulation & Zeroing Gradients

PyTorch does not overwrite gradients when `.backward()` is called multiple times; it adds (accumulates) them.

In [15]:
w = torch.tensor(2.0, requires_grad=True)

print("\n--- Experiment 4: Gradient Accumulation ---")

# Pass 1
loss1 = w * 3
loss1.backward()
print(f"After 1st backward pass (3 * 1): grad = {w.grad.item()}")

# Pass 2 without zeroing
loss2 = w * 3
loss2.backward()
print(f"After 2nd backward pass without zeroing: grad = {w.grad.item()}") # Should be 3 + 3 = 6

# Zeroing gradients
w.grad.zero_()
print(f"After w.grad.zero_(): grad = {w.grad.item()}")

# Pass 3 after zeroing
loss3 = w * 3
loss3.backward()
print(f"After 3rd backward pass post-zeroing: grad = {w.grad.item()}")


--- Experiment 4: Gradient Accumulation ---
After 1st backward pass (3 * 1): grad = 3.0
After 2nd backward pass without zeroing: grad = 6.0
After w.grad.zero_(): grad = 0.0
After 3rd backward pass post-zeroing: grad = 3.0


### Controlling Autograd: Standard vs. `torch.no_grad()` vs. `detach()`

When running inference, evaluation, or updating parameters manually, you need to disable graph construction to save memory and execution time.

In [16]:
x = torch.tensor(3.0, requires_grad=True)

# 1. Standard Autograd
y_standard = x * 2

# 2. torch.no_grad() Context
with torch.no_grad():
    y_nograd = x * 2

# 3. detach() Method
y_detached = (x * 2).detach()

print("\n--- Experiment 5: Autograd Control Modes ---")
print(f"Standard Execution:  requires_grad={y_standard.requires_grad}, grad_fn={y_standard.grad_fn}")
print(f"Inside no_grad():    requires_grad={y_nograd.requires_grad}, grad_fn={y_nograd.grad_fn}")
print(f"Using detach():      requires_grad={y_detached.requires_grad}, grad_fn={y_detached.grad_fn}")


--- Experiment 5: Autograd Control Modes ---
Standard Execution:  requires_grad=True, grad_fn=<MulBackward0 object at 0x000002026A89BF70>
Inside no_grad():    requires_grad=False, grad_fn=None
Using detach():      requires_grad=False, grad_fn=None


## What ChatGPT Expected Me to Learn from this Project

Project 13 was designed to make Autograd feel less magical.

By completing it, you should have understood that:

Forward computation
       →
Computational graph
       →
Loss
       →
Backward pass
       →
Gradients

is not an abstract idea anymore.

You have now seen the mechanism directly in PyTorch.

But we still built our examples from individual tensors.

The next step is abstraction.